# Product Confirmation Workflow - Direct S3 Access

This notebook reads DIST-ALERT products directly from S3 and runs the confirmation workflow without downloading.

In [1]:
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from dist_s1 import run_sequential_confirmation_of_dist_products_workflow

/Users/cmarshak/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
token = 'dist-event-10max_v1_32'
confirmed_products_dir = Path(f'confirmed_products_{token}')
confirmed_products_dir.mkdir(exist_ok=True)

## Load CSV with S3 URIs

In [3]:
csv_path = Path('dist-s1-events_v1_32_10max.csv')
df = pd.read_csv(csv_path)
print(f"Total products: {len(df)}")
df.head()

Total products: 854


,job_name,zip_url,s3_uri,browse_url,product_request_time,processing_duration,high_confidence_alert_threshold,mgrs_tile_id,post_date_buffer_days,stride_for_norm_param_estimation,...,memory_strategy,batch_size_for_norm_param_estimation,model_context_length,post_date,track_number,low_confidence_alert_threshold,n_workers_for_despeckling,device,max_pre_imgs_per_burst_mw,model_compilation
0,brazzaville_flood_and_landslides_2023__2023-12...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,s3://hyp3-tibet-jpl-test-contentbucket-hratibh...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-12-15T20:44:45+00:00,947.558,4.5,33MWR,1,23,...,high,32,10,2024-02-11,109,2.5,4,best,none,False
1,brazzaville_flood_and_landslides_2023__2023-12...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,s3://hyp3-tibet-jpl-test-contentbucket-hratibh...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-12-15T20:44:45+00:00,742.226,4.5,33MWR,1,23,...,high,32,10,2024-01-18,109,2.5,4,best,none,False
2,monkey_creek_fire_2024__2024-07-18__10TGQ__tra...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,s3://hyp3-tibet-jpl-test-contentbucket-hratibh...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-12-15T20:44:45+00:00,710.178,4.5,10TGQ,1,23,...,high,32,10,2024-08-28,42,2.5,4,best,none,False
3,monkey_creek_fire_2024__2024-07-18__10TGQ__tra...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,s3://hyp3-tibet-jpl-test-contentbucket-hratibh...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-12-15T20:44:45+00:00,575.377,4.5,10TGQ,1,23,...,high,32,10,2024-07-28,115,2.5,4,best,none,False
4,monkey_creek_fire_2024__2024-07-18__11TLK__tra...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,s3://hyp3-tibet-jpl-test-contentbucket-hratibh...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-12-15T20:44:45+00:00,1046.205,4.5,11TLK,1,23,...,high,32,10,2024-07-25,64,2.5,4,best,none,False


## Fix S3 URIs if needed

The s3_uri column may be missing a `/` after the bucket name. This cell fixes that.

In [4]:
def fix_s3_uri(uri):
    if not uri.startswith('s3://'):
        return uri
    
    # Extract bucket and key
    parts = uri.replace('s3://', '').split('/', 1)
    if len(parts) == 1:
        return uri
    
    bucket_and_prefix = parts[0]
    rest = parts[1] if len(parts) > 1 else ''
    
    # Find where the UUID starts (after bucket name)
    # Bucket name pattern: hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa
    # UUID pattern: 75556029-0ecc-4789-9d92-a0f147b075ed
    # Look for the pattern where bucket name ends
    import re
    # Match bucket name ending with 'contentbucket-' followed by random string
    match = re.match(r'(.*contentbucket-[a-z0-9]+)([a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12}.*)', bucket_and_prefix)
    
    if match:
        bucket = match.group(1)
        key = match.group(2)
        if rest:
            key = f"{key}/{rest}"
        return f"s3://{bucket}/{key}"
    
    return uri

df['s3_uri_fixed'] = df['s3_uri'].apply(fix_s3_uri)

# Check a few examples
print("Example S3 URI fixes:")
for i in range(min(3, len(df))):
    print(f"Original: {df.iloc[i]['s3_uri']}")
    print(f"Fixed:    {df.iloc[i]['s3_uri_fixed']}")
    print()

Example S3 URI fixes:
Original: s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa75556029-0ecc-4789-9d92-a0f147b075ed/OPERA_L3_DIST-ALERT-S1_T33MWR_20240211T043536Z_20251215T211147Z_S1A_30_v0.1
Fixed:    s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/75556029-0ecc-4789-9d92-a0f147b075ed/OPERA_L3_DIST-ALERT-S1_T33MWR_20240211T043536Z_20251215T211147Z_S1A_30_v0.1

Original: s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9paa94448fb-d140-4892-8e64-f0f802b23600/OPERA_L3_DIST-ALERT-S1_T33MWR_20240118T043537Z_20251215T210829Z_S1A_30_v0.1
Fixed:    s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/a94448fb-d140-4892-8e64-f0f802b23600/OPERA_L3_DIST-ALERT-S1_T33MWR_20240118T043537Z_20251215T210829Z_S1A_30_v0.1

Original: s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa500f9068-037c-4e0b-afcc-4e175123fd2a/OPERA_L3_DIST-ALERT-S1_T10TGQ_20240828T140625Z_20251215T210647Z_S1A_30_v0.1
Fixed:    s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/500f9068-037c-4e0b-afcc-4e175123fd2a/OPERA

## Group products by event and MGRS tile

In [5]:
# Extract event name from job_name
df['event'] = df['job_name'].str.split('__').str[0]

# Group by event and MGRS tile
grouped = df.groupby(['event', 'mgrs_tile_id'])

print(f"Number of event-tile combinations: {len(grouped)}")
print("\nEvent-tile combinations:")
for (event, tile), group_df in grouped:
    print(f"  {event} / {tile}: {len(group_df)} products")

Number of event-tile combinations: 36

Event-tile combinations:
  afghanistan_flood_2024 / 42SWF: 30 products
  attica_fire_2024 / 34SGH: 30 products
  bangladesh_coastal_flood_2024 / 45QYE: 20 products
  bangladesh_coastal_flood_2024 / 45QYF: 26 products
  bangladesh_coastal_flood_2024 / 45QZE: 24 products
  bangladesh_coastal_flood_2024 / 45QZF: 17 products
  bioko_fire_2024 / 32NMJ: 28 products
  brazzaville_flood_and_landslides_2023 / 33MWR: 15 products
  brazzaville_flood_and_landslides_2024 / 33MWR: 14 products
  chiapas_fire_2024 / 15QUU: 23 products
  chilcotin_river_landslide_and_flood_2024 / 10UEC: 14 products
  chile_fire_2024 / 19HBD: 22 products
  cipongkor_landslides_2024 / 48MYT: 14 products
  demak_flood_2024 / 49MDN: 22 products
  demak_flood_2024 / 49MEN: 23 products
  durkee_fire_2024 / 11TMJ: 39 products
  durkee_fire_2024 / 11TMK: 39 products
  hokkaido_landslides_2018 / 54TWN: 32 products
  los_angeles_fires_2025 / 11SLT: 30 products
  mai_mahiu_flood_and_landslid

## Run Confirmation Workflow

For each event-tile combination, run the sequential confirmation workflow using S3 URIs directly.

In [6]:
# Track results
results = []

for (event, tile), group_df in tqdm(list(grouped), desc="Processing event-tile combinations"):
    print(f"\nProcessing {event} / {tile} ({len(group_df)} products)")
    
    # Sort by post_date to ensure chronological order
    group_df = group_df.sort_values('post_date')
    
    # Get list of S3 product paths
    s3_product_paths = group_df['s3_uri_fixed'].tolist()
    
    if len(s3_product_paths) < 2:
        print(f"  Skipping {event}/{tile}: only {len(s3_product_paths)} product(s)")
        continue
    
    # Create output directory for this event-tile combination
    event_tile_dir = confirmed_products_dir / event / tile
    event_tile_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        # Run confirmation workflow
        run_sequential_confirmation_of_dist_products_workflow(
            dist_s1_data=s3_product_paths,
            dst_dist_product_parent=event_tile_dir,
            tqdm_enabled=False
        )
        
        results.append({
            'event': event,
            'tile': tile,
            'num_products': len(s3_product_paths),
            'status': 'success',
            'error': None
        })
        print(f"  ✓ Completed {event}/{tile}")
        
    except Exception as e:
        results.append({
            'event': event,
            'tile': tile,
            'num_products': len(s3_product_paths),
            'status': 'failed',
            'error': str(e)
        })
        print(f"  ✗ Failed {event}/{tile}: {e}")

Processing event-tile combinations:   0%|                        | 0/36 [00:00<?, ?it/s]


Processing afghanistan_flood_2024 / 42SWF (30 products)


/Users/cmarshak/bekaert-team/dist-s1/src/dist_s1/data_models/output_models.py:273: UserWarning: Layer GEN-DIST-STATUS does not exist: s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/56d2095c-0b3f-4f69-b894-1cb53c2f40aa/OPERA_L3_DIST-ALERT-S1_T42SWF_20240414T013129Z_20251215T205716Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T42SWF_20240414T013129Z_20251215T205716Z_S1A_30_v0.1_GEN-DIST-STATUS.tif
  warn(f'Layer {layer} does not exist: {path_or_uri}', UserWarning)
/Users/cmarshak/bekaert-team/dist-s1/src/dist_s1/data_models/output_models.py:273: UserWarning: Layer GEN-METRIC does not exist: s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/56d2095c-0b3f-4f69-b894-1cb53c2f40aa/OPERA_L3_DIST-ALERT-S1_T42SWF_20240414T013129Z_20251215T205716Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T42SWF_20240414T013129Z_20251215T205716Z_S1A_30_v0.1_GEN-METRIC.tif
  warn(f'Layer {layer} does not exist: {path_or_uri}', UserWarning)
/Users/cmarshak/bekaert-team/dist-s1/src/dist_s1/data_models/output_models.py:273: U

  ✗ Failed afghanistan_flood_2024/42SWF: S3 product missing required layers: s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/56d2095c-0b3f-4f69-b894-1cb53c2f40aa/OPERA_L3_DIST-ALERT-S1_T42SWF_20240414T013129Z_20251215T205716Z_S1A_30_v0.1

Processing attica_fire_2024 / 34SGH (30 products)


/Users/cmarshak/bekaert-team/dist-s1/src/dist_s1/data_models/output_models.py:273: UserWarning: Layer GEN-DIST-STATUS does not exist: s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/cf854914-b6e2-4134-88d8-67f722d4be8d/OPERA_L3_DIST-ALERT-S1_T34SGH_20240715T162359Z_20251215T210724Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T34SGH_20240715T162359Z_20251215T210724Z_S1A_30_v0.1_GEN-DIST-STATUS.tif
  warn(f'Layer {layer} does not exist: {path_or_uri}', UserWarning)
/Users/cmarshak/bekaert-team/dist-s1/src/dist_s1/data_models/output_models.py:273: UserWarning: Layer GEN-METRIC does not exist: s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/cf854914-b6e2-4134-88d8-67f722d4be8d/OPERA_L3_DIST-ALERT-S1_T34SGH_20240715T162359Z_20251215T210724Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T34SGH_20240715T162359Z_20251215T210724Z_S1A_30_v0.1_GEN-METRIC.tif
  warn(f'Layer {layer} does not exist: {path_or_uri}', UserWarning)
/Users/cmarshak/bekaert-team/dist-s1/src/dist_s1/data_models/output_models.py:273: U

...

  ✗ Failed bangladesh_coastal_flood_2024/45QZE: S3 product missing required layers: s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/24a9b697-3918-4c48-b5d4-dc776f9b6da6/OPERA_L3_DIST-ALERT-S1_T45QZE_20240610T120431Z_20251215T210555Z_S1A_30_v0.1

Processing bangladesh_coastal_flood_2024 / 45QZF (17 products)


/Users/cmarshak/bekaert-team/dist-s1/src/dist_s1/data_models/output_models.py:273: UserWarning: Layer GEN-DIST-STATUS does not exist: s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/98e321a4-6910-4a78-807b-3394b1c1ff47/OPERA_L3_DIST-ALERT-S1_T45QZF_20240610T120440Z_20251215T211114Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T45QZF_20240610T120440Z_20251215T211114Z_S1A_30_v0.1_GEN-DIST-STATUS.tif
  warn(f'Layer {layer} does not exist: {path_or_uri}', UserWarning)
/Users/cmarshak/bekaert-team/dist-s1/src/dist_s1/data_models/output_models.py:273: UserWarning: Layer GEN-METRIC does not exist: s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/98e321a4-6910-4a78-807b-3394b1c1ff47/OPERA_L3_DIST-ALERT-S1_T45QZF_20240610T120440Z_20251215T211114Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T45QZF_20240610T120440Z_20251215T211114Z_S1A_30_v0.1_GEN-METRIC.tif
  warn(f'Layer {layer} does not exist: {path_or_uri}', UserWarning)
/Users/cmarshak/bekaert-team/dist-s1/src/dist_s1/data_models/output_models.py:273: U

  ✗ Failed bangladesh_coastal_flood_2024/45QZF: S3 product missing required layers: s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/98e321a4-6910-4a78-807b-3394b1c1ff47/OPERA_L3_DIST-ALERT-S1_T45QZF_20240610T120440Z_20251215T211114Z_S1A_30_v0.1

Processing bioko_fire_2024 / 32NMJ (28 products)


/Users/cmarshak/bekaert-team/dist-s1/src/dist_s1/data_models/output_models.py:273: UserWarning: Layer GEN-DIST-STATUS does not exist: s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/31623e80-3a33-4979-88dd-e4f44e030ccc/OPERA_L3_DIST-ALERT-S1_T32NMJ_20240112T174457Z_20251215T210313Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T32NMJ_20240112T174457Z_20251215T210313Z_S1A_30_v0.1_GEN-DIST-STATUS.tif
  warn(f'Layer {layer} does not exist: {path_or_uri}', UserWarning)
/Users/cmarshak/bekaert-team/dist-s1/src/dist_s1/data_models/output_models.py:273: UserWarning: Layer GEN-METRIC does not exist: s3://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa/31623e80-3a33-4979-88dd-e4f44e030ccc/OPERA_L3_DIST-ALERT-S1_T32NMJ_20240112T174457Z_20251215T210313Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T32NMJ_20240112T174457Z_20251215T210313Z_S1A_30_v0.1_GEN-METRIC.tif
  warn(f'Layer {layer} does not exist: {path_or_uri}', UserWarning)
/Users/cmarshak/bekaert-team/dist-s1/src/dist_s1/data_models/output_models.py:273: U

KeyboardInterrupt: 

## Summary

In [ ]:
results_df = pd.DataFrame(results)
print("\nConfirmation Results:")
print(f"  Total combinations processed: {len(results_df)}")
print(f"  Successful: {(results_df['status'] == 'success').sum()}")
print(f"  Failed: {(results_df['status'] == 'failed').sum()}")

if (results_df['status'] == 'failed').any():
    print("\nFailed combinations:")
    failed = results_df[results_df['status'] == 'failed']
    for _, row in failed.iterrows():
        print(f"  {row['event']}/{row['tile']}: {row['error']}")

results_df

In [ ]:
# List confirmed products
confirmed_products = list(confirmed_products_dir.rglob('OPERA_L3_DIST-ALERT-S1*/'))
print(f"\nTotal confirmed products created: {len(confirmed_products)}")
print("\nFirst 10 confirmed products:")
for prod in confirmed_products[:10]:
    print(f"  {prod.relative_to(confirmed_products_dir)}")